# Final-evaluation rerun: Catboost

This notebook reruns the established tuning procedure on the new frozen development split created by notebook 10. It never loads the locked final-test rows. It selects validation thresholds for 70%, 75%, 80%, 85%, and 90% target recall, then saves the frozen model and artifacts for notebook 17.

# CatBoost Model Optimisation

This notebook keeps the same final comparison design used for the other tuned models:

1. Model settings are selected without using the threshold-validation or final-test sets.
2. The probability threshold is selected only on the validation set.
3. The final test set is evaluated once after the model settings and threshold are fixed.
4. The validation recall target is 80%.

The main methodological difference is that CatBoost receives the original categorical variables directly. They are not one-hot encoded before training. This allows CatBoost to use its own categorical-feature statistics and categorical combinations.

Like the XGBoost notebook, an internal early-stopping split is used to choose the learning rate and number of boosting rounds. After those choices are fixed, the final CatBoost model is refitted on the complete 64% model-training set.

In [1]:
import json

In [2]:
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import sklearn
import catboost

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import (
    ParameterSampler,
    StratifiedKFold,
    train_test_split
)

print("scikit-learn version:", sklearn.__version__)
print("CatBoost version:", catboost.__version__)

scikit-learn version: 1.9.0
CatBoost version: 1.2.10


## 1. Load the cleaned dataset and create the output folder

In [3]:
# Locate the cleaned dataset and save this final rerun separately from earlier experiments.
PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = (
        parent
        / "Processed_Dataset"
        / "diabetic_data_cleaned_stage1.csv"
    )

    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find "
        "Processed_Dataset/diabetic_data_cleaned_stage1.csv"
    )

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Model_Results"
    / "catboost_optimisation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(DATA_PATH)

print("Dataset path:")
print(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

df.head()

# Final-evaluation artifacts are kept separate from the earlier development runs.
FINAL_EVALUATION_DIR = PROJECT_ROOT / "Final_Evaluation"
SPLIT_DIR = FINAL_EVALUATION_DIR / "Data_Splits"
OUTPUT_DIR = FINAL_EVALUATION_DIR / "Model_Artifacts" / "catboost"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nFinal-evaluation output directory:")
print(OUTPUT_DIR)


Dataset path:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Processed_Dataset/diabetic_data_cleaned_stage1.csv

Dataset shape:
(69987, 56)

Final-evaluation output directory:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Final_Evaluation/Model_Artifacts/catboost


## 2. Select the same modelling features used by the other tuned models

For a fair model comparison, this notebook uses exactly the same 18 predictors as the existing tuned models.

In [4]:
# Use the same 18 predictors as the other final models.
target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

model_features = (
    categorical_features
    + numeric_features
)

missing_features = [
    feature
    for feature in model_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "These modelling features are missing: "
        f"{missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].astype(int).copy()

print("X shape:")
print(X.shape)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget counts:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))

X shape:
(69987, 18)

Features used:
['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target counts:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64


## 3. CatBoost-specific feature preparation

CatBoost handles categorical variables directly, so this notebook deliberately does not use `OneHotEncoder`.

Categorical missing values are replaced by the literal category `"Missing"` and categorical columns are converted to strings. Numeric columns are converted to numeric values; numeric missing values can remain as `NaN` for CatBoost to handle internally.

This preparation does not learn anything from the outcome or from validation/test data, so it does not create leakage.

In [5]:
def prepare_catboost_dataframe(
    X_data,
    categorical_features,
    numeric_features
):
    """Prepare features for CatBoost's native categorical handling.

    Categorical variables remain categorical strings rather than being one-hot
    encoded. Numerical variables are coerced to numeric so CatBoost receives a
    consistent DataFrame.
    """

    X_prepared = X_data.copy()

    for feature in categorical_features:
        values = X_prepared[feature].astype("object")
        values = values.where(
            values.notna(),
            "Missing"
        )
        X_prepared[feature] = values.astype(str)

    for feature in numeric_features:
        X_prepared[feature] = pd.to_numeric(
            X_prepared[feature],
            errors="coerce"
        )

    return X_prepared


X = prepare_catboost_dataframe(
    X,
    categorical_features=categorical_features,
    numeric_features=numeric_features
)

print("Categorical dtypes:")
print(X[categorical_features].dtypes)

print("\nNumeric dtypes:")
print(X[numeric_features].dtypes)

print("\nMissing values after preparation:")
print(X.isna().sum())

Categorical dtypes:
gender                     object
race_group                 object
age_group                  object
admission_source_group     object
discharge_group            object
medical_specialty_group    object
primary_diagnosis          object
hba1c_group                object
max_glu_serum              object
diabetesMed                object
dtype: object

Numeric dtypes:
time_in_hospital      int64
num_lab_procedures    int64
num_procedures        int64
num_medications       int64
number_outpatient     int64
number_emergency      int64
number_inpatient      int64
number_diagnoses      int64
dtype: object

Missing values after preparation:
gender                     0
race_group                 0
age_group                  0
admission_source_group     0
discharge_group            0
medical_specialty_group    0
primary_diagnosis          0
hba1c_group                0
max_glu_serum              0
diabetesMed                0
time_in_hospital           0
num_lab_procedure

In [6]:
forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = (
    forbidden_features
    .intersection(X.columns)
)

assert not unexpected_features, (
    "Unexpected or potentially leaking features found: "
    f"{unexpected_features}"
)

assert not X.columns.duplicated().any(), (
    "Duplicate column names were found in X."
)

assert len(X) == len(y), (
    "X and y contain different numbers of rows."
)

assert y.isna().sum() == 0, (
    "The target contains missing values."
)

assert set(y.unique()).issubset({0, 1}), (
    "The target must contain only 0 and 1."
)

for feature in categorical_features:
    assert X[feature].isna().sum() == 0, (
        f"Categorical feature {feature} still contains missing values."
    )

print("Feature and target checks passed.")

Feature and target checks passed.


## 4. Create development, validation, test, and early-stopping sets

The final model-training, validation, and test proportions remain 60%, 20%, and 20% of the full dataset.

An internal early-stopping set is taken from the 64% model-training portion. It is used only to choose the learning rate and number of boosting rounds. It is not the validation set used later for threshold selection.

In [7]:

# Load only the frozen model-training and threshold-validation rows.
# The locked final-test row file is intentionally not read in this notebook.

SPLIT_DIR = PROJECT_ROOT / "Final_Evaluation" / "Data_Splits"

train_split_path = SPLIT_DIR / "model_train_rows.csv"
validation_split_path = SPLIT_DIR / "threshold_validation_rows.csv"

if not train_split_path.exists() or not validation_split_path.exists():
    raise FileNotFoundError(
        "Final split files are missing. Run 10_create_final_split.ipynb first."
    )

train_idx = (
    pd.read_csv(train_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)
val_idx = (
    pd.read_csv(validation_split_path)["row_position"]
    .astype(int)
    .to_numpy()
)

assert set(train_idx).isdisjoint(set(val_idx))

X_model_train = X.iloc[train_idx].copy()
y_model_train = y.iloc[train_idx].copy()

X_val = X.iloc[val_idx].copy()
y_val = y.iloc[val_idx].copy()

split_summary = pd.DataFrame({
    "split": ["model_training", "threshold_validation"],
    "rows": [len(X_model_train), len(X_val)],
    "positive_count": [int(y_model_train.sum()), int(y_val.sum())],
    "positive_rate": [float(y_model_train.mean()), float(y_val.mean())],
})

print(
    "Final-test rows have NOT been loaded. "
    "They stay locked until notebook 17."
)
split_summary

# Split only the model-training rows again to create an internal early-stopping subset.
X_search_train, X_early_stop, y_search_train, y_early_stop = train_test_split(
    X_model_train,
    y_model_train,
    test_size=0.20,
    stratify=y_model_train,
    random_state=42
)

internal_summary = pd.DataFrame({
    "split": [
        "CV/search training",
        "internal early stopping",
        "full model training after selection",
        "threshold validation",
    ],
    "rows": [
        len(X_search_train),
        len(X_early_stop),
        len(X_model_train),
        len(X_val),
    ],
    "positive_rate": [
        float(y_search_train.mean()),
        float(y_early_stop.mean()),
        float(y_model_train.mean()),
        float(y_val.mean()),
    ],
})

internal_summary


Final-test rows have NOT been loaded. They stay locked until notebook 17.


,split,rows,positive_rate
0,CV/search training,33592,0.089813
1,internal early stopping,8399,0.089773
2,full model training after selection,41991,0.089805
3,threshold validation,13998,0.089799


## 5. Evaluation and threshold-selection helper functions

In [8]:
def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model"
):
    """Evaluate one operating point from predicted readmission probabilities.

    The probability threshold converts risk scores into 0/1 predictions.
    The returned dictionary combines ranking/probability metrics with
    confusion-matrix and workload measures used in the final comparison.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    # Convert continuous probabilities into binary predictions at this threshold.
    y_pred = (
        y_proba >= threshold
    ).astype(int)

    # Extract TN/FP/FN/TP once so several threshold-based metrics can reuse them.
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )

    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )

    # Workload measure: how many patients are flagged for each true readmission found.
    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    return {
        "model": model_name,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0
        ),
        "auroc": roc_auc_score(
            y_true,
            y_proba
        ),
        "auprc": average_precision_score(
            y_true,
            y_proba
        ),
        "brier_score": brier_score_loss(
            y_true,
            y_proba
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "predicted_positive": int(predicted_positive),
        "predicted_negative": int(predicted_negative),
        "predicted_positive_rate": predicted_positive_rate,
        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        )
    }

In [9]:
def confusion_matrix_from_proba(
    y_true,
    y_proba,
    threshold=0.5
):
    """Create a labelled confusion matrix from probabilities."""

    y_pred = (
        np.asarray(y_proba) >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    return pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted"
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted"
        ]
    )

In [10]:
def threshold_sweep(
    y_true,
    y_proba,
    model_name="Model",
    thresholds=None
):
    """Calculate performance across probability thresholds."""

    # The regular grid is descriptive only; the locked threshold is selected separately.
    if thresholds is None:
        thresholds = np.round(
            np.arange(
                0.01,
                0.951,
                0.01
            ),
            2
        )

    results = [
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold,
            model_name=model_name
        )
        for threshold in thresholds
    ]

    return pd.DataFrame(results)

In [11]:
def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model"
):
    """
    Select the threshold with the lowest false-positive rate
    among thresholds that achieve the required recall.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    # Generate every available ROC operating point for threshold selection.
    false_positive_rates, recalls, thresholds = roc_curve(
        y_true,
        y_proba,
        drop_intermediate=False
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": false_positive_rates,
        "specificity": 1 - false_positive_rates
    })

    candidate_table = candidate_table[
        np.isfinite(
            candidate_table["threshold"]
        )
    ].copy()

    # Keep only thresholds that satisfy the requested minimum recall.
    eligible_candidates = candidate_table[
        candidate_table["recall"] >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            "No threshold achieved recall >= "
            f"{min_recall:.2f}."
        )

    # Among eligible thresholds, minimise FPR; if tied, prefer the higher threshold.
    eligible_candidates = (
        eligible_candidates
        .sort_values(
            by=[
                "false_positive_rate",
                "threshold"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        eligible_candidates.iloc[0]["threshold"]
    )

    # Recalculate the complete metric set at the selected operating point.
    selected_metrics = evaluate_predictions_from_proba(
        y_true=y_true,
        y_proba=y_proba,
        threshold=selected_threshold,
        model_name=model_name
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates
    )

## 6. Stage 1: tune CatBoost structure, regularisation, categorical combinations, sampling, and class weighting

The first search deliberately keeps `iterations=400` and `learning_rate=0.05` fixed. This lets us compare different tree structures and regularisation settings without mixing the question of *what kind of trees to build* with *how long boosting should continue*.

A manual five-fold cross-validation loop is used so every candidate is ranked by the same scikit-learn average precision (AUPRC) measure used in the other optimisation notebooks, while CatBoost still receives the original categorical columns directly.



In [12]:
RECALL_TARGET = 0.80
# Stage 1 samples 30 structural/regularisation configurations.
STRUCTURE_SEARCH_ITERATIONS = 30
STRUCTURE_SEARCH_TREES = 400
STRUCTURE_SEARCH_LEARNING_RATE = 0.05

# Manual five-fold CV ranks structural candidates without touching validation rows.
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Include the observed negative:positive ratio as a data-driven class-weight candidate.
negative_count = int((y_search_train == 0).sum())
positive_count = int((y_search_train == 1).sum())
imbalance_ratio = negative_count / positive_count

print(
    f"Required validation recall: "
    f"{RECALL_TARGET:.0%}"
)
print("Cross-validation folds:", cross_validation.n_splits)
print("Structure search candidates:", STRUCTURE_SEARCH_ITERATIONS)
print("Negative to positive ratio:", imbalance_ratio)

Required validation recall: 80%
Cross-validation folds: 5
Structure search candidates: 30
Negative to positive ratio: 10.134239310573417


In [13]:
# Stage 1 tunes tree depth, regularisation, categorical-combination complexity,
# row sampling and class weighting while the boosting schedule remains fixed.
catboost_structure_param_distributions = {
    # Symmetric-tree depth controls interaction complexity.
    "depth": [
        4,
        5,
        6,
        7,
        8,
        9,
        10
    ],

    # L2 regularisation shrinks leaf values.
    "l2_leaf_reg": [
        1.0,
        3.0,
        5.0,
        10.0,
        20.0,
        50.0
    ],

    "random_strength": [
        0.0,
        0.5,
        1.0,
        2.0,
        5.0
    ],

    "border_count": [
        64,
        128,
        254
    ],

    # Controls complexity of automatically constructed categorical combinations.
    "max_ctr_complexity": [
        1,
        2,
        3,
        4
    ],

    "subsample": [
        0.60,
        0.80,
        1.00
    ],

    "scale_pos_weight": [
        1.0,
        2.0,
        4.0,
        6.0,
        float(imbalance_ratio)
    ]
}

sampled_structure_parameters = list(
    ParameterSampler(
        param_distributions=(
            catboost_structure_param_distributions
        ),
        n_iter=STRUCTURE_SEARCH_ITERATIONS,
        random_state=42
    )
)

len(sampled_structure_parameters)

30

In [14]:
def build_catboost_classifier(
    structure_params,
    iterations,
    learning_rate,
    thread_count=-1
):
    """Create one CatBoost classifier from a candidate configuration.

    Centralising model construction keeps the loss, PRAUC evaluation metric,
    categorical-feature list, bootstrap mode and random seed consistent.
    """

    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC:type=Classic",
        iterations=iterations,
        learning_rate=learning_rate,
        cat_features=categorical_features,
        boosting_type="Plain",
        bootstrap_type="MVS",
        random_seed=42,
        thread_count=thread_count,
        allow_writing_files=False,
        verbose=False,
        **structure_params
    )

In [15]:
# Manual CV explicitly trains every sampled CatBoost structure across the same five folds.
catboost_cv_candidate_rows = []
catboost_cv_fold_rows = []

# Outer loop: one structural candidate at a time.
for candidate_number, structure_params in enumerate(
    sampled_structure_parameters,
    start=1
):
    print(
        f"Candidate {candidate_number}/"
        f"{STRUCTURE_SEARCH_ITERATIONS}"
    )

    fold_train_auprc = []
    fold_validation_auprc = []
    fold_validation_auroc = []

    # Inner loop: rotate that candidate through the five stratified folds.
    for fold_number, (
        fold_train_indices,
        fold_validation_indices
    ) in enumerate(
        cross_validation.split(
            X_search_train,
            y_search_train
        ),
        start=1
    ):
        X_fold_train = X_search_train.iloc[
            fold_train_indices
        ]
        y_fold_train = y_search_train.iloc[
            fold_train_indices
        ]

        X_fold_validation = X_search_train.iloc[
            fold_validation_indices
        ]
        y_fold_validation = y_search_train.iloc[
            fold_validation_indices
        ]

        fold_model = build_catboost_classifier(
            structure_params=structure_params,
            iterations=STRUCTURE_SEARCH_TREES,
            learning_rate=(
                STRUCTURE_SEARCH_LEARNING_RATE
            ),
            thread_count=-1
        )

        fold_model.fit(
            X_fold_train,
            y_fold_train,
            verbose=False
        )

        train_proba = (
            fold_model
            .predict_proba(X_fold_train)[:, 1]
        )

        validation_proba = (
            fold_model
            .predict_proba(
                X_fold_validation
            )[:, 1]
        )

        train_auprc = average_precision_score(
            y_fold_train,
            train_proba
        )

        validation_auprc = average_precision_score(
            y_fold_validation,
            validation_proba
        )

        validation_auroc = roc_auc_score(
            y_fold_validation,
            validation_proba
        )

        fold_train_auprc.append(
            train_auprc
        )
        fold_validation_auprc.append(
            validation_auprc
        )
        fold_validation_auroc.append(
            validation_auroc
        )

        catboost_cv_fold_rows.append({
            "candidate": candidate_number,
            "fold": fold_number,
            **structure_params,
            "train_auprc": train_auprc,
            "validation_auprc": validation_auprc,
            "validation_auroc": validation_auroc
        })

    mean_train_auprc = float(
        np.mean(fold_train_auprc)
    )

    mean_validation_auprc = float(
        np.mean(fold_validation_auprc)
    )

    catboost_cv_candidate_rows.append({
        "candidate": candidate_number,
        **structure_params,
        "mean_train_auprc": mean_train_auprc,
        "std_train_auprc": float(
            np.std(
                fold_train_auprc,
                ddof=1
            )
        ),
        "mean_validation_auprc": (
            mean_validation_auprc
        ),
        "std_validation_auprc": float(
            np.std(
                fold_validation_auprc,
                ddof=1
            )
        ),
        "mean_validation_auroc": float(
            np.mean(fold_validation_auroc)
        ),
        "std_validation_auroc": float(
            np.std(
                fold_validation_auroc,
                ddof=1
            )
        ),
        "train_validation_auprc_gap": (
            mean_train_auprc
            - mean_validation_auprc
        )
    })

catboost_structure_cv_results = pd.DataFrame(
    catboost_cv_candidate_rows
)

catboost_structure_cv_fold_results = pd.DataFrame(
    catboost_cv_fold_rows
)

catboost_structure_cv_results = (
    catboost_structure_cv_results
    .sort_values(
        by=[
            "mean_validation_auprc",
            "mean_validation_auroc",
            "train_validation_auprc_gap"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

catboost_structure_cv_results[
    "rank_validation_auprc"
] = (
    catboost_structure_cv_results[
        "mean_validation_auprc"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

catboost_structure_cv_results.to_csv(
    OUTPUT_DIR
    / "catboost_structure_cv_results.csv",
    index=False
)

catboost_structure_cv_fold_results.to_csv(
    OUTPUT_DIR
    / "catboost_structure_cv_fold_results.csv",
    index=False
)

catboost_structure_cv_results.head(20)

Candidate 1/30
Candidate 2/30
Candidate 3/30
Candidate 4/30
Candidate 5/30
Candidate 6/30
Candidate 7/30
Candidate 8/30
Candidate 9/30
Candidate 10/30
Candidate 11/30
Candidate 12/30
Candidate 13/30
Candidate 14/30
Candidate 15/30
Candidate 16/30
Candidate 17/30
Candidate 18/30
Candidate 19/30
Candidate 20/30
Candidate 21/30
Candidate 22/30
Candidate 23/30
Candidate 24/30
Candidate 25/30
Candidate 26/30
Candidate 27/30
Candidate 28/30
Candidate 29/30
Candidate 30/30


,candidate,subsample,scale_pos_weight,random_strength,max_ctr_complexity,l2_leaf_reg,depth,border_count,mean_train_auprc,std_train_auprc,mean_validation_auprc,std_validation_auprc,mean_validation_auroc,std_validation_auroc,train_validation_auprc_gap,rank_validation_auprc
0,21,1.0,1.000000,2.0,1,10.0,5,64,0.188625,0.005314,0.143908,0.003523,0.628850,0.005221,0.044717,1
1,8,0.6,2.000000,2.0,2,50.0,5,128,0.178628,0.003504,0.143796,0.003723,0.628589,0.005171,0.034831,2
2,9,1.0,2.000000,1.0,3,50.0,4,64,0.167324,0.003928,0.143583,0.003612,0.628367,0.003149,0.023741,3
3,2,1.0,2.000000,1.0,4,5.0,4,64,0.183734,0.005013,0.143522,0.003002,0.627096,0.005949,0.040212,4
4,1,0.6,1.000000,2.0,3,20.0,5,128,0.174765,0.003413,0.143486,0.003855,0.628049,0.004910,0.031279,5
5,12,0.8,1.000000,5.0,3,50.0,6,64,0.172218,0.005414,0.143330,0.002636,0.627770,0.002115,0.028889,6
6,17,0.6,1.000000,1.0,3,3.0,5,254,0.196072,0.005907,0.143321,0.003374,0.627723,0.005165,0.052751,7
7,29,0.8,1.000000,1.0,4,50.0,5,64,0.173421,0.004699,0.143200,0.003944,0.627253,0.004477,0.030221,8
8,26,0.8,4.000000,5.0,1,20.0,4,64,0.173491,0.001425,0.142890,0.003341,0.628211,0.005191,0.030601,9
9,11,0.6,2.000000,1.0,1,5.0,5,64,0.220546,0.005043,0.142863,0.004610,0.627025,0.004523,0.077683,10


## 7. Select and inspect the strongest CatBoost structure candidate

In [16]:
best_structure_row = (
    catboost_structure_cv_results
    .iloc[0]
)

selected_structure_params = {
    "depth": int(
        best_structure_row["depth"]
    ),
    "l2_leaf_reg": float(
        best_structure_row["l2_leaf_reg"]
    ),
    "random_strength": float(
        best_structure_row["random_strength"]
    ),
    "border_count": int(
        best_structure_row["border_count"]
    ),
    "max_ctr_complexity": int(
        best_structure_row["max_ctr_complexity"]
    ),
    "subsample": float(
        best_structure_row["subsample"]
    ),
    "scale_pos_weight": float(
        best_structure_row["scale_pos_weight"]
    )
}

print("Selected CatBoost structure parameters:")
for parameter, value in selected_structure_params.items():
    print(f"{parameter}: {value}")

print("\nBest cross-validation AUPRC:")
print(
    best_structure_row[
        "mean_validation_auprc"
    ]
)

print("\nTrain-validation AUPRC gap:")
print(
    best_structure_row[
        "train_validation_auprc_gap"
    ]
)

Selected CatBoost structure parameters:
depth: 5
l2_leaf_reg: 10.0
random_strength: 2.0
border_count: 64
max_ctr_complexity: 1
subsample: 1.0
scale_pos_weight: 1.0

Best cross-validation AUPRC:
0.14390790690033645

Train-validation AUPRC gap:
0.04471666130938365


## 8. Stage 2: choose the learning rate and number of boosting rounds with early stopping

Each learning-rate candidate is allowed to build up to 2,000 trees. Training stops after 75 rounds without improvement in CatBoost's validation PRAUC. We then calculate scikit-learn AUPRC, AUROC, and Brier score on the same internal early-stopping set and select the learning-rate candidate with the highest scikit-learn AUPRC.

The separate threshold-validation set remains untouched.

In [17]:
# Stage 2 fixes the winning structure, compares learning rates and uses early stopping
# to choose the number of boosting iterations.
LEARNING_RATE_CANDIDATES = [
    0.02,
    0.03,
    0.05,
    0.08,
    0.10
]

MAX_BOOSTING_ROUNDS = 2000
EARLY_STOPPING_ROUNDS = 75

learning_rate_results = []
early_stopped_models = {}

for learning_rate in LEARNING_RATE_CANDIDATES:
    candidate_model = build_catboost_classifier(
        structure_params=(
            selected_structure_params
        ),
        iterations=MAX_BOOSTING_ROUNDS,
        learning_rate=learning_rate,
        thread_count=-1
    )

    # The internal early-stop subset determines when additional trees stop improving PRAUC.
    candidate_model.fit(
        X_search_train,
        y_search_train,
        eval_set=(
            X_early_stop,
            y_early_stop
        ),
        use_best_model=True,
        early_stopping_rounds=(
            EARLY_STOPPING_ROUNDS
        ),
        verbose=False
    )

    early_stop_proba = (
        candidate_model
        .predict_proba(X_early_stop)[:, 1]
    )

    selected_tree_count = int(
        candidate_model.tree_count_
    )

    best_iteration = int(
        candidate_model.get_best_iteration()
    )

    learning_rate_results.append({
        "learning_rate": learning_rate,
        "selected_iterations": (
            selected_tree_count
        ),
        "best_iteration_zero_based": (
            best_iteration
        ),
        "early_stop_auprc": average_precision_score(
            y_early_stop,
            early_stop_proba
        ),
        "early_stop_auroc": roc_auc_score(
            y_early_stop,
            early_stop_proba
        ),
        "early_stop_brier_score": brier_score_loss(
            y_early_stop,
            early_stop_proba
        )
    })

    early_stopped_models[
        learning_rate
    ] = candidate_model

learning_rate_results = (
    pd.DataFrame(learning_rate_results)
    .sort_values(
        by=[
            "early_stop_auprc",
            "early_stop_auroc",
            "selected_iterations"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

learning_rate_results.to_csv(
    OUTPUT_DIR
    / "catboost_learning_rate_early_stopping_results.csv",
    index=False
)

learning_rate_results

,learning_rate,selected_iterations,best_iteration_zero_based,early_stop_auprc,early_stop_auroc,early_stop_brier_score
0,0.03,172,171,0.152577,0.620158,0.080117
1,0.08,113,112,0.152278,0.618799,0.080043
2,0.10,105,104,0.152077,0.618899,0.080047
3,0.02,370,369,0.151718,0.620283,0.080075
4,0.05,106,105,0.151271,0.619715,0.080113


In [18]:
selected_learning_rate = float(
    learning_rate_results.iloc[0][
        "learning_rate"
    ]
)

selected_iterations = int(
    learning_rate_results.iloc[0][
        "selected_iterations"
    ]
)

print("Selected learning rate:")
print(selected_learning_rate)

print("\nSelected number of boosting rounds:")
print(selected_iterations)

Selected learning rate:
0.03

Selected number of boosting rounds:
172


## 9. Refit the selected CatBoost model on the complete model-training set

The internal early-stopping data now returns to the training data. The final selected CatBoost model is fitted on the complete 64% model-training set using the chosen structure, learning rate, and number of trees. The threshold-validation and final-test sets are still untouched.

In [19]:
# Refit a fresh CatBoost model on all frozen model-training rows after structure and
# boosting duration have been selected.
final_catboost_params = deepcopy(
    selected_structure_params
)

best_catboost_model = build_catboost_classifier(
    structure_params=final_catboost_params,
    iterations=selected_iterations,
    learning_rate=selected_learning_rate,
    thread_count=-1
)

best_catboost_model.fit(
    X_model_train,
    y_model_train,
    verbose=False
)

best_catboost_model

CatBoostClassifier(allow_writing_files=False, boosting_type='Plain', bootstrap_type='MVS', border_count=64, cat_features=['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed'], depth=5, eval_metric='PRAUC:type=Classic', iterations=172, l2_leaf_reg=10.0, learning_rate=0.03, loss_function='Logloss', max_ctr_complexity=1, random_seed=42, random_strength=2.0, scale_pos_weight=1.0, subsample=1.0, verbose=False)

In [20]:
catboost_structure_summary = pd.Series({
    "number_of_boosting_rounds": int(
        best_catboost_model.tree_count_
    ),
    "learning_rate": float(
        selected_learning_rate
    ),
    "depth": selected_structure_params[
        "depth"
    ],
    "l2_leaf_reg": selected_structure_params[
        "l2_leaf_reg"
    ],
    "random_strength": selected_structure_params[
        "random_strength"
    ],
    "border_count": selected_structure_params[
        "border_count"
    ],
    "max_ctr_complexity": selected_structure_params[
        "max_ctr_complexity"
    ],
    "subsample": selected_structure_params[
        "subsample"
    ],
    "scale_pos_weight": selected_structure_params[
        "scale_pos_weight"
    ],
    "boosting_type": "Plain",
    "bootstrap_type": "MVS"
})

catboost_structure_summary.to_csv(
    OUTPUT_DIR
    / "catboost_selected_structure_summary.csv",
    header=["value"]
)

catboost_structure_summary

number_of_boosting_rounds      172
learning_rate                 0.03
depth                            5
l2_leaf_reg                   10.0
random_strength                2.0
border_count                    64
max_ctr_complexity               1
subsample                      1.0
scale_pos_weight               1.0
boosting_type                Plain
bootstrap_type                 MVS
dtype: object

## 10. Evaluate the selected model on the validation set at the default threshold

In [21]:
# Threshold selection and probability-distribution checks use the frozen validation rows only.
y_val_proba_catboost = (
    best_catboost_model
    .predict_proba(X_val)[:, 1]
)

print(
    "Minimum validation probability:",
    y_val_proba_catboost.min()
)

print(
    "Maximum validation probability:",
    y_val_proba_catboost.max()
)

print(
    "Number of unique validation probabilities:",
    np.unique(y_val_proba_catboost).size
)

Minimum validation probability: 0.0511758477085799
Maximum validation probability: 0.37247616080010554
Number of unique validation probabilities: 13941


In [22]:
catboost_probability_summary = (
    pd.DataFrame({
        "actual_class": np.asarray(y_val),
        "predicted_readmission_probability": (
            y_val_proba_catboost
        )
    })
    .groupby("actual_class")
    ["predicted_readmission_probability"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

catboost_probability_summary.to_csv(
    OUTPUT_DIR
    / "catboost_validation_probability_summary.csv"
)

catboost_probability_summary

,count,mean,std,min,10%,25%,50%,75%,90%,max
actual_class,,,,,,,,,,
0,12741.0,0.091035,0.028156,0.051176,0.063554,0.069275,0.080787,0.111620,0.125573,0.372476
1,1257.0,0.105610,0.037354,0.055805,0.067544,0.075865,0.104088,0.121765,0.140958,0.346056


In [23]:
catboost_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_catboost,
        threshold=0.5,
        model_name=(
            "CatBoost validation default"
        )
    )
)

pd.Series(
    catboost_val_default_results
)

model                                          CatBoost validation default
threshold                                                              0.5
accuracy                                                          0.910201
precision                                                              0.0
recall                                                                 0.0
specificity                                                            1.0
false_positive_rate                                                    0.0
false_negative_rate                                                    1.0
f1                                                                     0.0
f2                                                                     0.0
auroc                                                             0.627648
auprc                                                             0.147833
brier_score                                                       0.080223
true_negative            

## 11. Select the validation threshold that reaches at least 80% recall

Among all validation thresholds that achieve the recall target, the rule selects the threshold with the lowest false-positive rate. This is the same operating-point rule used for the other tuned models.

In [24]:
(
    catboost_selected_threshold,
    catboost_val_selected_results,
    catboost_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_catboost,
    min_recall=RECALL_TARGET,
    model_name=(
        "CatBoost validation selected"
    )
)

print("Selected CatBoost threshold:")
print(catboost_selected_threshold)

pd.DataFrame([
    catboost_val_default_results,
    catboost_val_selected_results
])

Selected CatBoost threshold:
0.07320609583799173


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,CatBoost validation default,0.500000,0.910201,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,...,0.147833,0.080223,12741,0,1257,0,0,13998,0.000000,NaN
1,CatBoost validation selected,0.073206,0.397985,0.109562,0.800318,0.358292,0.641708,0.199682,0.192739,0.353976,...,0.147833,0.080223,4565,8176,251,1006,9182,4816,0.655951,9.127237


In [25]:
catboost_eligible_thresholds.to_csv(
    OUTPUT_DIR
    / "catboost_eligible_validation_thresholds.csv",
    index=False
)

catboost_eligible_thresholds.head(20)

,threshold,recall,false_positive_rate,specificity
0,0.073206,0.800318,0.641708,0.358292
1,0.073204,0.800318,0.641786,0.358214
2,0.073202,0.800318,0.641865,0.358135
3,0.073199,0.800318,0.641943,0.358057
4,0.073196,0.800318,0.642022,0.357978
5,0.073191,0.800318,0.642100,0.357900
6,0.073188,0.800318,0.642179,0.357821
7,0.073187,0.801114,0.642179,0.357821
8,0.073184,0.801114,0.642257,0.357743
9,0.073184,0.801114,0.642336,0.357664


In [26]:
# Save threshold sensitivity without changing the primary >=80% recall operating rule.
catboost_threshold_sweep = threshold_sweep(
    y_true=y_val,
    y_proba=y_val_proba_catboost,
    model_name="CatBoost validation"
)

catboost_threshold_sweep.to_csv(
    OUTPUT_DIR
    / "catboost_validation_threshold_sweep.csv",
    index=False
)

catboost_threshold_sweep[
    [
        "threshold",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f1",
        "f2",
        "true_positive",
        "false_positive",
        "false_negative",
        "predicted_positive_rate"
    ]
]

,threshold,recall,precision,specificity,false_positive_rate,f1,f2,true_positive,false_positive,false_negative,predicted_positive_rate
0,0.01,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
1,0.02,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
2,0.03,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
3,0.04,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
4,0.05,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
90,0.91,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
91,0.92,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
92,0.93,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
93,0.94,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0


In [27]:
comparison_columns = [
    "model",
    "threshold",
    "auprc",
    "auroc",
    "brier_score",
    "accuracy",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "false_negative_rate",
    "f1",
    "f2",
    "true_positive",
    "true_negative",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found"
]

catboost_validation_comparison = pd.DataFrame([
    catboost_val_default_results,
    catboost_val_selected_results
])

catboost_validation_comparison = (
    catboost_validation_comparison[
        comparison_columns
    ]
)

catboost_validation_comparison.to_csv(
    OUTPUT_DIR
    / "catboost_validation_results.csv",
    index=False
)

catboost_validation_comparison

,model,threshold,auprc,auroc,brier_score,accuracy,recall,precision,specificity,false_positive_rate,false_negative_rate,f1,f2,true_positive,true_negative,false_positive,false_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,CatBoost validation default,0.500000,0.147833,0.627648,0.080223,0.910201,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0,12741,0,1257,0.000000,NaN
1,CatBoost validation selected,0.073206,0.147833,0.627648,0.080223,0.397985,0.800318,0.109562,0.358292,0.641708,0.199682,0.192739,0.353976,1006,4565,8176,251,0.655951,9.127237


In [28]:
catboost_val_selected_cm = (
    confusion_matrix_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_catboost,
        threshold=(
            catboost_selected_threshold
        )
    )
)

catboost_val_selected_cm.to_csv(
    OUTPUT_DIR
    / "catboost_validation_confusion_matrix.csv"
)

catboost_val_selected_cm

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4565,8176
Actual readmitted,251,1006


In [29]:
y_val_pred_catboost = (
    y_val_proba_catboost
    >= catboost_selected_threshold
).astype(int)

print(
    classification_report(
        y_val,
        y_val_pred_catboost,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)

                precision    recall  f1-score   support

Not readmitted       0.95      0.36      0.52     12741
    Readmitted       0.11      0.80      0.19      1257

      accuracy                           0.40     13998
     macro avg       0.53      0.58      0.36     13998
  weighted avg       0.87      0.40      0.49     13998



## Freeze final development-stage artifacts

In [30]:
# Freeze several recall operating points and row-level validation predictions for notebook 17.

RECALL_TARGETS = [0.70, 0.75, 0.80, 0.85, 0.90]

threshold_rows = []

for recall_target in RECALL_TARGETS:
    selected_threshold, selected_metrics, _ = (
        choose_threshold_for_minimum_recall(
            y_true=y_val,
            y_proba=y_val_proba_catboost,
            min_recall=recall_target,
            model_name="CatBoost validation"
        )
    )

    threshold_rows.append({
        "target_recall": recall_target,
        "selected_threshold": selected_threshold,
        "validation_recall": selected_metrics["recall"],
        "validation_precision": selected_metrics["precision"],
        "validation_specificity": selected_metrics["specificity"],
        "validation_false_positive_rate": selected_metrics["false_positive_rate"],
        "validation_f2": selected_metrics["f2"],
        "validation_true_positive": selected_metrics["true_positive"],
        "validation_false_negative": selected_metrics["false_negative"],
        "validation_false_positive": selected_metrics["false_positive"],
        "validation_true_negative": selected_metrics["true_negative"],
        "validation_flagged_rate": selected_metrics["predicted_positive_rate"],
    })

selected_thresholds = pd.DataFrame(threshold_rows)

selected_thresholds.to_csv(
    OUTPUT_DIR / "selected_validation_thresholds.csv",
    index=False
)

validation_predictions = pd.DataFrame({
    "row_position": val_idx,
    "y_true": np.asarray(y_val, dtype=int),
    "probability": np.asarray(y_val_proba_catboost, dtype=float),
})

validation_predictions.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

selected_thresholds

best_catboost_model.save_model(str(OUTPUT_DIR / "final_model.cbm"))

with open(OUTPUT_DIR / "selected_hyperparameters.json", "w") as f:
    json.dump(
        {
            "structure": selected_structure_params,
            "learning_rate": selected_learning_rate,
            "iterations": selected_iterations,
        },
        f,
        indent=2,
        default=str,
    )

print("Saved frozen CatBoost model and artifacts.")


Saved frozen CatBoost model and artifacts.
